# Desafio Flor de Íris (Parte 2) — *One-Hot Encoding* e Rede Neural `MLP`

## 1. O que é *One-Hot Encoding* e por que usamos 3 Neurônios na Saída?

Se tentássemos ensinar uma única saída numérica da rede a responder `0` para *Setosa*, `1` para *Versicolor* e `2` para *Virginica*, estaríamos forçando uma **relação falsa de grandeza**: a rede acharia que *Versicolor* é a "média aritmética" entre *Setosa* e *Virginica*!

Em classificação multiclasse com **Perceptron Multicamadas (PMC / MLP)**, usamos **One-Hot Encoding** ("um fio ligado por vez"): colocamos **3 neurônios na camada de saída** ($y_1, y_2, y_3$):

| Espécie | $d_1$ | $d_2$ | $d_3$ |
| :--- | :---: | :---: | :---: |
| `Iris-setosa` | `1` | `0` | `0` |
| `Iris-versicolor` | `0` | `1` | `0` |
| `Iris-virginica` | `0` | `0` | `1` |

*Analogia de Painel Industrial:* Em vez de um único voltímetro tentando apontar 3 rótulos diferentes, instalamos **3 lâmpadas sinalizadoras independentes** no painel — a lâmpada que brilhar mais forte (`np.argmax`) indica a espécie diagnosticada!

---

## O Desafio (Parte 2)
Construa e treine um **MLP** (em `Scikit-Learn` ou `Python/NumPy Puro` com *Backpropagation* e vetores de *bias*) usando `../arquivos-csv/iris_train.csv` e salve as predições de `../arquivos-csv/iris_exercise.csv`. Pratique abaixo antes de conferir os gabaritos!

In [ ]:
# Escreva sua implementação de MLP aqui!

<details>
<summary><b>Clique aqui para ver o Gabarito 2.1 — MLP Multiclasse com Scikit-Learn (Comentado)</b></summary>

```python
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# 1. Carrega treino e teste
train = pd.read_csv("../arquivos-csv/iris_train.csv")
test = pd.read_csv("../arquivos-csv/iris_exercise.csv")

features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X_train, y_train = train[features], train["class"]
X_test = test[features]

# 2. Padronização das 4 medidas geométricas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Rede Neural com 2 camadas ocultas (10 e 10 neurônios); o Scikit-Learn aplica
# One-Hot Encoding e ativação Softmax na saída automaticamente para 3 classes!
mlp = MLPClassifier(hidden_layer_sizes=(10, 10), max_iter=2000, random_state=42)
mlp.fit(X_train_scaled, y_train)
print(f"Acurácia no treino: {mlp.score(X_train_scaled, y_train) * 100:.2f}%")

# 4. Previsão e exportação para CSV
y_pred = mlp.predict(X_test_scaled)
pd.DataFrame({"id": test["id"], "class": y_pred}).to_csv("solution_iris_mlp_sklearn.csv", index=False)
print("Arquivo gerado: solution_iris_mlp_sklearn.csv")
```

</details>

<details>
<summary><b>Clique aqui para ver o Gabarito 2.2 — MLP com Backpropagation e Bias em NumPy Puro (Comentado)</b></summary>

```python
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Carregamento e Padronização Z-score
train = pd.read_csv("../arquivos-csv/iris_train.csv")
test = pd.read_csv("../arquivos-csv/iris_exercise.csv")

features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X_train_raw = train[features].to_numpy()
y_train_raw = train["class"].to_numpy()
X_test_raw = test[features].to_numpy()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# 2. Construção explícita da matriz One-Hot Encoding (N x 3) na ordem alfabética das classes
classes = sorted(list(np.unique(y_train_raw)))
Y_onehot = np.zeros((len(y_train_raw), len(classes)))
for idx, nome_classe in enumerate(y_train_raw):
    Y_onehot[idx, classes.index(nome_classe)] = 1.0

class MLPIrisPuro:
    '''Perceptron Multicamadas 4 -> 12 -> 3 com vetores de bias (b1, b2) e Backpropagation.'''
    def __init__(self, d_in, d_hid, d_out, lr=0.5, seed=42):
        rng = np.random.default_rng(seed)
        self.W1 = rng.uniform(-0.5, 0.5, size=(d_in, d_hid))  # Pesos Entrada -> Oculta (4x12)
        self.b1 = np.zeros((1, d_hid))                        # Bias da camada oculta (1x12)
        self.W2 = rng.uniform(-0.5, 0.5, size=(d_hid, d_out)) # Pesos Oculta -> Saída (12x3)
        self.b2 = np.zeros((1, d_out))                        # Bias da camada de saída (1x3)
        self.lr = lr

    @staticmethod
    def _sigmoide(u):
        return 1.0 / (1.0 + np.exp(-np.clip(u, -50.0, 50.0)))

    def treinar(self, X, Y_desejado, epocas=2500):
        N = X.shape[0]
        for _ in range(epocas):
            # Forward Pass: propaga entradas até as 3 lâmpadas de saída
            h = self._sigmoide(X @ self.W1 + self.b1)         # (N x 12)
            o = self._sigmoide(h @ self.W2 + self.b2)         # (N x 3)

            # Backward Pass: calcula gradientes locais pela derivada o*(1 - o)
            delta_o = (Y_desejado - o) * (o * (1.0 - o))      # (N x 3)
            delta_h = (delta_o @ self.W2.T) * (h * (1.0 - h)) # (N x 12)

            # Atualização dos pesos e dos biases dividida por N (média estável por amostra)
            self.W2 += (self.lr / N) * (h.T @ delta_o)
            self.b2 += (self.lr / N) * np.sum(delta_o, axis=0, keepdims=True)
            self.W1 += (self.lr / N) * (X.T @ delta_h)
            self.b1 += (self.lr / N) * np.sum(delta_h, axis=0, keepdims=True)

    def prever_indices(self, X):
        h = self._sigmoide(X @ self.W1 + self.b1)
        o = self._sigmoide(h @ self.W2 + self.b2)
        # Escolhe a coluna (0, 1 ou 2) com maior ativação em cada linha
        return np.argmax(o, axis=1)

# 3. Treinamento e geração do arquivo de saída
rede_iris = MLPIrisPuro(d_in=4, d_hid=12, d_out=3, lr=0.8, seed=42)
rede_iris.treinar(X_train_scaled, Y_onehot, epocas=3000)

acc_tr = np.mean(rede_iris.prever_indices(X_train_scaled) == np.argmax(Y_onehot, axis=1)) * 100.0
print(f"Acurácia no treino do MLP Puro (com bias): {acc_tr:.2f}%")

indices_previstos = rede_iris.prever_indices(X_test_scaled)
nomes_previstos = [classes[i] for i in indices_previstos]

pd.DataFrame({"id": test["id"], "class": nomes_previstos}).to_csv("solution_iris_mlp_puro.csv", index=False)
print("Arquivo gerado: solution_iris_mlp_puro.csv")
```

</details>